### Сборка пака фотографий

Итак, в этом ноутбуке мы будем загружать фото, обрабатывать их и упаковывать в один файл cian_photos_224.npz в сжатом виде, чтобы в дальнейшем передавать его в основной ноутбук с обучением модели

In [ ]:
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
import os, io, glob, requests, zipfile
from concurrent.futures import ThreadPoolExecutor

Задаем конфигурацию: качество сжатия изображения =88, порцию обработки (в случае если Colab вылетит) =400

In [ ]:
PHOTOS_PUBLIC_LINK = 'https://disk.360.yandex.ru/d/SvsxBvG_-EWgtw'
PHOTOS_DIR = 'photos_new'
SIZE = 224
QUALITY = 88
SHARD_LISTINGS = 400
SHARD_DIR = 'shards'
os.makedirs(SHARD_DIR, exist_ok=True)
OUT_PACK = 'cian_photos_224.npz'

Функция, которая отправляет api запрос в яндекс диск, чтобы получить прямую ссылку для скачивания всего архива c фотками:

In [ ]:
def yadisk_direct_url(public_link):
  api = 'https://cloud-api.yandex.net/v1/disk/public/resources/download'
  r = requests.get(api, params={'public_key': public_link})
  r.raise_for_status()
  return r.json()['href']

Скачиваем архив по прямой ссылке с докачкой на случай обрыва сети, после чего распаковываем его:

In [1]:
if not os.path.isdir(PHOTOS_DIR):
  print('Пошло-поехало...')
  href = yadisk_direct_url(PHOTOS_PUBLIC_LINK)
  !wget -c -q -O photos.zip '$href'
  print('Распаковка...')
  with zipfile.ZipFile('photos.zip') as z:
    z.extractall('.')

Считаем кол-во уникальных папок с объявлениями:

In [ ]:
n_listings = len([d for d in os.listdir(PHOTOS_DIR) if os.path.isdir(os.path.join(PHOTOS_DIR, d))])
n_listings

Отсортированный список со всеми объявлениями:

In [ ]:
listings = sorted(d for d in os.listdir(PHOTOS_DIR) if os.path.isdir(os.path.join(PHOTOS_DIR, d)))

Формируем список путей к каждой фотографии в датасете, привязывая каждое фото к id объявления:

In [3]:
def listing_files(lid):
  d = os.path.join(PHOTOS_DIR, lid)
  files = sorted(f for f in os.listdir(d) if f.lower().endswith(('.jpg','.jpeg','.png')))
  return [(os.path.join(d, f), int(lid)) for f in files]

Полученный массив нарезаем на шарды:

In [ ]:
shards = [listings[i:i+SHARD_LISTINGS] for i in range(0, len(listings), SHARD_LISTINGS)]

Функция для превращения фотографии в квадрат без искажений пропрций. Также, чтобы было пошустрей, используем оптимизацию libpeg, которая снижает нагрузку на процессор:

In [2]:
def pad_resize_encode(path, size=SIZE, quality=QUALITY):
  try:
    with open(path, 'rb') as f:
      img = Image.open(io.BytesIO(f.read()))
      img.draft('RGB', (size,size))
      img = img.convert('RGB')
  except Exception:
    img = Image.new('RGB', (size,size), (0,0,0))
  w, h = img.size
  side = max(w,h)
  canvas = Image.new('RGB', (side,side), (0,0,0))
  canvas.paste(img, ((side - w)//2, (side - h)//2))
  canvas = canvas.resize((size, size), Image.BILINEAR)
  buf = io.BytesIO()
  canvas.save(buf, 'JPEG', quality=quality)
  return buf.getvalue()

Выполняем параллельную обработку фтографий: берем шард, собираем все пути к фотографиям и запускаем сжатие в многопоточке. Результат упкаковываем в отдельный файл shard_*.npz в виде blob'а (т.е. много подряд идущих байтов, кодирующих изображения) и карты смещений offsets (границы байтов по каждой картинке):

In [ ]:
def process_shard(shard_id, shard_listings):
  out_path = os.path.join(SHARD_DIR, f'shard_{shard_id:04d}.npz')
  if os.path.exists(out_path):
    return out_path
  tasks = []
  for lid in shard_listings:
    tasks += listing_files(lid)
  jpegs = [None] * len(tasks)
  with ThreadPoolExecutor(max_workers=16) as ex:
    for k, b in enumerate(ex.map(lambda t: pad_resize_encode(t[0]), tasks)):
      jpegs[k] = b
  lids = np.array([t[1] for t in tasks], dtype=np.int64)
  sizes = np.array([len(b) for b in jpegs], dtype=np.int64)
  offsets = np.zeros(len(jpegs)+1, dtype=np.int64)
  offsets[1:] = np.cumsum(sizes)
  blob = np.frombuffer(b''.join(jpegs), dtype=np.uint8)
  np.savez(out_path, blob=blob, offsets=offsets, listing_ids=lids)
  return out_path

По очереди подаем шарды:

In [ ]:
for sid, sl in enumerate(tqdm(shards, desc='шарды')):
  process_shard(sid, sl)

Собираем все шарды в один файл cian_photos_224.npz:

In [4]:
shard_paths = sorted(glob.glob(os.path.join(SHARD_DIR, 'shard_*.npz')))
blobs, all_lids, all_off = [], [], [np.array([0], dtype=np.int64)]
base = 0
for p in tqdm(shard_paths, desc='merge'):
  z = np.load(p)
  blobs.append(z['blob'])
  all_lids.append(z['listing_ids'])
  all_off.append(z['offsets'][1:] + base)
  base += int(z['blob'].shape[0])

Склеиваем все в итоговые масивы, проверяя целостность данных:

In [ ]:
blob = np.concatenate(blobs)
offsets = np.concatenate(all_off)
listing_ids = np.concatenate(all_lids)
assert offsets[-1] == blob.shape[0] and len(offsets) == len(listing_ids) + 1
np.savez(OUT_PACK, blob=blob, offsets=offsets, listing_ids=listing_ids)

Сохраняем на диск:

In [5]:
from google.colab import drive
import shutil

In [ ]:
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/cian', exist_ok=True)
dst = os.path.join('/content/drive/MyDrive/cian', OUT_PACK)
shutil.copy(OUT_PACK, dst)